# Create compound-subsets


In [ ]:
import random
import os
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text
import scipy.stats as stats
import csv

In [ ]:
def load_csv(file_path: str) -> list:
    data = []
    with open(file_path, 'r') as f:
        csv_reader = csv.DictReader(f)
        header = csv_reader.fieldnames
        for row in csv_reader:
            data.append(row)
    return data

def load_ratings(file_path:str) -> dict:
    csv_data = load_csv(file_path)
    head_ratings = dict() # structure: key = (compound,head), val = mean rating
    modifier_ratings = dict() # structure: key = (compound,modifier), val = mean rating
    all_ratings = dict() # structure: key = (compound,constituent), val = mean rating
    for rating in csv_data:
        compound = rating["compound"]
        modifier,head = compound.split()
        if "const" in rating: # file with human annotations
            if rating["const"] == modifier:
                modifier_ratings[(compound,modifier)] = rating["mean"]
            else:
                head_ratings[(compound,head)] = rating["mean"]
            all_ratings[(compound,rating["const"])] = rating["mean"]
        else: # file with measured scores between embeddings
            modifier_ratings[(compound,modifier)] = rating["modifier_score"]
            head_ratings[(compound,head)] = rating["head_score"]
            all_ratings[(compound,modifier)] = rating["modifier_score"]
            all_ratings[(compound,head)] = rating["head_score"]
    return {"heads":head_ratings,"modifiers":modifier_ratings,"all":all_ratings}

In [ ]:
# load compounds and constituents
compounds = set()
constituents = set()
with open("/path/to/repo/Data/compounds.csv","r") as infile:
    for line in infile:
        if not line.startswith("ID"):
            compound = line.split(",")[1]
            compounds.add(compound.strip())
            for c in compound.split(" "):
                constituents.add(c.strip())
print(len(compounds),compounds)
print(len(constituents),constituents)

200 {'wedding day', 'paper loss', 'independence day', 'construction paper', 'ice queen', 'prom queen', 'sheet music', 'radio station', 'strategy game', 'culture critic', 'ballot paper', 'emergency surgery', 'capital equipment', 'way station', 'hill station', 'farewell party', 'movie critic', 'camping area', 'image quality', 'horror fan', 'wedding cake', 'dog dinner', 'rowing machine', 'fruit cake', 'fantasy romance', 'family romance', 'heart surgery', 'safety helmet', 'war movie', 'horror game', 'family dinner', 'memory card', 'recording equipment', 'road closure', 'eye surgery', 'factory closure', 'loan agreement', 'spirit photography', 'studio album', 'space helmet', 'street party', 'health service', 'material processing', 'puppy dog', 'privacy protection', 'food processing', 'glove box', 'fraud scheme', 'shoe box', 'hype machine', 'plea agreement', 'carnival queen', 'data quality', 'tea service', 'suggestion box', 'blood loss', 'propaganda machine', 'wedding photography', 'ballet sh

In [4]:
sorted_compounds = sorted(compounds,key=lambda x: (x.split()[1],x.split()[0]))
print(sorted_compounds)

['confidentiality agreement', 'licensing agreement', 'loan agreement', 'plea agreement', 'service agreement', 'tenancy agreement', 'concept album', 'family album', 'music album', 'studio album', 'camping area', 'conservation area', 'dining area', 'language area', 'parking area', 'service area', 'entry barrier', 'language barrier', 'noise barrier', 'pain barrier', 'safety barrier', 'ticket barrier', 'glove box', 'letter box', 'lunch box', 'memory box', 'selection box', 'shoe box', 'suggestion box', 'tool box', 'cup cake', 'fruit cake', 'novelty cake', 'potato cake', 'rice cake', 'wedding cake', 'bank card', 'gift card', 'greeting card', 'identity card', 'library card', 'loyalty card', 'memory card', 'emergency closure', 'factory closure', 'road closure', 'case conference', 'party conference', 'peace conference', 'web conference', 'art critic', 'culture critic', 'drama critic', 'food critic', 'movie critic', 'music critic', 'assessment day', 'family day', 'field day', 'game day', 'indepe

In [ ]:
subset_path = "/path/to/repo/Data/Subsets/"

### Create subset with unique heads

For each head: randomly choose a modifier

In [ ]:
# extract list of compound where the head always differ -> choose compounds randomly
head_sorted_compounds = dict() # key = head, value = list of compounds with this head
for compound in compounds:
    mod,head = compound.split(" ")
    if head in head_sorted_compounds:
        head_sorted_compounds[head] += [compound]
    else:
        head_sorted_compounds[head] = [compound]
print(len(head_sorted_compounds)," different heads are available, namely ",sorted(list(head_sorted_compounds.keys())))
compounds_unique_heads = []
for head,compound_candidates in head_sorted_compounds.items():
    compounds_unique_heads += [random.choice(compound_candidates)]
print(len(compounds_unique_heads),compounds_unique_heads)

# save randomly chosen compounds to file
with open(subset_path+"compounds_unique-heads.csv","w") as outfile:
    outfile.write("ID,noun_compound,component_1,component_2\n")
    for i, comp in enumerate(compounds_unique_heads):
        mod,head = comp.split(" ")
        outfile.write(str(i)+","+comp+","+mod.strip()+","+head.strip()+"\n")

36  different heads are available, namely  ['agreement', 'album', 'area', 'barrier', 'box', 'cake', 'card', 'closure', 'conference', 'critic', 'day', 'dinner', 'dog', 'equipment', 'fan', 'food', 'game', 'helmet', 'loss', 'machine', 'movie', 'music', 'paper', 'party', 'photography', 'processing', 'protection', 'quality', 'queen', 'romance', 'scheme', 'service', 'shoe', 'station', 'surgery', 'trip']
36 ['summer romance', 'noise barrier', 'image quality', 'licensing agreement', 'ballot paper', 'macro photography', 'camping area', 'web conference', 'puppy dog', 'horror fan', 'room service', 'material processing', 'gangster movie', 'way station', 'space helmet', 'ticket machine', 'memory card', 'health protection', 'hen party', 'art critic', 'emergency surgery', 'dog food', 'family album', 'ice queen', 'pension scheme', 'ballet shoe', 'game day', 'country music', 'field trip', 'laboratory equipment', 'ball game', 'wine dinner', 'lunch box', 'blood loss', 'factory closure', 'fruit cake']


### Create subsets for different concreteness groups

In [ ]:
# load file with concreteness values
compound_info = pd.read_csv("/path/to/repo/Data/compounds_details.tsv",sep="\t")

In [ ]:
concreteness_grouped = compound_info.groupby("category")["compound"].apply(list).to_dict()
for group,comps in concreteness_grouped.items():
    print(group,len(comps),comps)

AA 16 ['confidentiality agreement', 'licensing agreement', 'plea agreement', 'service agreement', 'tenancy agreement', 'emergency closure', 'memory loss', 'emotion processing', 'health protection', 'privacy protection', 'service quality', 'fantasy romance', 'fraud scheme', 'incentive scheme', 'emergency service', 'health service']
AC 49 ['concept album', 'conservation area', 'language area', 'service area', 'language barrier', 'safety barrier', 'memory box', 'selection box', 'suggestion box', 'novelty cake', 'identity card', 'loyalty card', 'memory card', 'peace conference', 'culture critic', 'drama critic', 'assessment day', 'independence day', 'farewell dinner', 'service dog', 'emergency equipment', 'safety equipment', 'horror fan', 'convenience food', 'health food', 'soul food', 'elimination game', 'horror game', 'strategy game', 'safety helmet', 'hype machine', 'propaganda machine', 'horror movie', 'romance movie', 'mood music', 'soul music', 'consultation paper', 'identity paper',

In [ ]:
for group,comps in concreteness_grouped.items():
    # save grouped compounds to file
    with open(subset_path+"compounds_"+group+".csv","w") as outfile:
        outfile.write("ID,noun_compound,component_1,component_2\n")
        for i, comp in enumerate(comps):
            mod,head = comp.split(" ")
            outfile.write(str(i)+","+comp+","+mod.strip()+","+head.strip()+"\n")

### Create head-specific subsets (e.g. all compounds with "dog" as head)

In [5]:
# group the compounds together by head
head_sorted_compounds = dict() # key = head, value = list of compounds with this head
for compound in compounds:
    mod,head = compound.split(" ")
    if head in head_sorted_compounds:
        head_sorted_compounds[head] += [compound]
    else:
        head_sorted_compounds[head] = [compound]
print(len(head_sorted_compounds)," different heads are available, namely ",sorted(list(head_sorted_compounds.keys())))

36  different heads are available, namely  ['agreement', 'album', 'area', 'barrier', 'box', 'cake', 'card', 'closure', 'conference', 'critic', 'day', 'dinner', 'dog', 'equipment', 'fan', 'food', 'game', 'helmet', 'loss', 'machine', 'movie', 'music', 'paper', 'party', 'photography', 'processing', 'protection', 'quality', 'queen', 'romance', 'scheme', 'service', 'shoe', 'station', 'surgery', 'trip']


In [ ]:
for head,compounds_with_this_head in head_sorted_compounds.items():
    # save grouped compounds to file
    with open(subset_path+"compounds_head_"+head+".csv","w") as outfile:
        outfile.write("ID,noun_compound,component_1,component_2\n")
        for i, comp in enumerate(compounds_with_this_head):
            mod,head = comp.split(" ")
            outfile.write(str(i)+","+comp+","+mod.strip()+","+head.strip()+"\n")